# Benchmark Construction Pipeline

In [17]:
import json, math, re, random
import pandas as pd, numpy as np
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
import time
from pathlib import Path
import itertools
from tools import filter_data, exclude, count_by, sentiment_breakdown, add_shares, add_priority, apply_min_volume, rank_top, share_of, two_prop_test, chi_squared, sample_reviews

pd.set_option("display.max_colwidth", None)   # None = don't truncate cell values (show full column text)
random.seed(29)

In [18]:
OUT_DIR = Path("../benchmark_outputs") 
OUT_DIR.mkdir(parents=True, exist_ok=True)

# one timestamp per run; prefixes the corpus + task files so runs don't overwrite each other
RUN_TS = time.strftime("%Y-%m-%d_%H%M%S")

FABSA = pd.read_csv("clean_fabsa.csv")
SENTS = ["positive", "negative", "neutral"]

In [19]:
# llm setup
load_dotenv()
client = OpenAI()
MODEL = "vertex_ai/gemini-2.5-flash" 

In [20]:
# cost logging
PRICE_IN, PRICE_OUT = 0.30/1e6, 2.50/1e6     # USD/token — verify current gemini-flash pricing
USAGE = defaultdict(lambda: {"calls":0, "in":0, "out":0})
USAGE.clear()


def gemini(prompt, block, max_tokens=2000, response_format=None):
    kwargs = dict(model=MODEL,
                  messages=[{"role": "user", "content": prompt}],
                  max_tokens=max_tokens)
    if response_format:
        kwargs["response_format"] = response_format
    r = client.chat.completions.create(**kwargs)
    u = r.usage
    USAGE[block]["calls"] += 1
    USAGE[block]["in"] += u.prompt_tokens
    USAGE[block]["out"] += u.completion_tokens
    choice = r.choices[0]
    if getattr(choice, "finish_reason", None) == "length":
        raise ValueError(f"[{block}] response truncated at max_tokens={max_tokens} "
                         f"(generated {u.completion_tokens} tokens) -- raise max_tokens or batch the call")
    return choice.message.content


# helper - call gemini, strip json fences, parse JSON

def _extract_json_payload(text):
    if isinstance(text, (dict, list)):
        return text
    if text is None:
        raise ValueError("empty response")
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()
    if not cleaned:
        raise ValueError("empty response")

    # Remove fenced code blocks even if the model adds a language tag.
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r"\s*```$", "", cleaned, flags=re.IGNORECASE | re.MULTILINE).strip()

    # Normalise smart SINGLE quotes only (apostrophes). Do NOT touch smart double quotes: they are
    # JSON string delimiters, so rewriting a curly double-quote inside answer text to a straight one
    # turns a valid payload into an unescaped-quote parse error.
    cleaned = cleaned.replace("‘", "'").replace("’", "'")

    # strict=False tolerates raw newlines/tabs the model may leave inside long answer strings.
    try:
        return json.loads(cleaned, strict=False)
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder(strict=False)
    for start in range(len(cleaned)):
        if cleaned[start] not in "{[":
            continue
        try:
            obj, end = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            continue
        if end > 0:
            return obj

    # show head AND tail: a clean head with a cut-off tail means the response was truncated.
    head = cleaned[:150].replace("\n", " ")
    tail = cleaned[-150:].replace("\n", " ")
    raise ValueError(f"could not parse JSON payload ({len(cleaned)} chars) -- if the tail looks "
                     f"cut off the response was truncated. head: {head!r} ... tail: {tail!r}")


def gemini_json(prompt, block, max_tokens=6000, retries=3):
    last_error = None
    for attempt in range(retries):
        txt = gemini(
            prompt,
            block,
            max_tokens,
            response_format={"type": "json_object"},
        )
        try:
            return _extract_json_payload(txt)
        except ValueError as exc:
            last_error = exc
            if attempt < retries - 1:
                continue
    raise ValueError(f"[{block}] invalid JSON response after {retries} attempts: {last_error}") from last_error


def cost_report():
    tot=0
    for b,d in USAGE.items():
        c=d["in"]*PRICE_IN+d["out"]*PRICE_OUT; tot+=c
        print(f"{b:12} calls={d['calls']:3d}  in={d['in']:7d}  out={d['out']:7d}  ${c:.4f}")
    print(f"{'TOTAL':12} {'':22} ${tot:.4f}")

# Step 1: plant insights by defining groups

In [21]:
# define aspect hierarchy
FABSA_INDUSTRIES = sorted(FABSA.industry.unique())

HIERARCHY = {
    "account-management": ["account-access"],
    "company-brand":      ["competitor", "general-satisfaction", "reviews"],
    "logistics-ride":     ["speed"],
    "online-experience":  ["app-website"],
    "booking-experience": ["ease-of-use"],
    "staff-support":      ["attitude-of-staff", "phone", "email"],
    "value":              ["discounts-promotions", "price-value-for-money"],
}

In [22]:
# sample 3 industries and 5 aspects

def sample_aspects(target=5):
    """Pick parents one by one, take all their children, stop when we have enough."""
    parents = list(HIERARCHY.keys())
    random.shuffle(parents)
    picked_parents, children = [], []
    for p in parents:
        picked_parents.append(p)
        children += HIERARCHY[p]
        if len(children) >= target:
            children = children[:target]
            break
    return picked_parents, children

# --- SMOKE TEST: shrink scope for a fast end-to-end run. Restore for the full build. ---
# Keep >=2 industries so the L4/L5 comparison + two-proportion tasks have a pair to work with.
# INDUSTRIES = random.sample(FABSA_INDUSTRIES, 3)
# PARENTS_USED, ASPECTS = sample_aspects(5)
INDUSTRIES = random.sample(FABSA_INDUSTRIES, 2)
PARENTS_USED, ASPECTS = sample_aspects(2)

print("industries:", INDUSTRIES)
print("parents:", PARENTS_USED, "-> aspects:", ASPECTS)

industries: ['Trading', 'Consulting']
parents: ['online-experience', 'company-brand'] -> aspects: ['app-website', 'competitor']


In [23]:
# standardised org names per industry, e.g. Banking -> [BankA, BankB], Information Technology -> [ITA, ITB]
ORG_PREFIX = {
    "Banking": "Bank", "Consulting": "Consult", "Fashion": "Fashion", "Groceries": "Grocer",
    "Information Technology": "IT", "Price Comparison": "Price", "Ride Hailing": "Ride",
    "Streaming": "Stream", "Trading": "Trade", "Travel Booking": "Travel",
}
ORGS_PER_INDUSTRY = 2
ORGS = {ind: [f"{ORG_PREFIX[ind]}{chr(65 + i)}" for i in range(ORGS_PER_INDUSTRY)] for ind in INDUSTRIES}

for ind in INDUSTRIES:
    print(f"{ind:25s} -> {ORGS[ind]}")

Trading                   -> ['TradeA', 'TradeB']
Consulting                -> ['ConsultA', 'ConsultB']


## Set corpus size

In [24]:
# check there are enough seed examples for each cell
CELLS_needed = [(ind, asp, sent) for ind in INDUSTRIES for asp in ASPECTS for sent in SENTS]
for ind, asp, sent in CELLS_needed:
    k = len(FABSA[(FABSA.industry==ind)&(FABSA.child_aspect==asp)&(FABSA.sentiment==sent)])
    if k < 5:
        print(f"thin seed: {ind}/{asp}/{sent} = {k}")

thin seed: Trading/app-website/neutral = 4
thin seed: Trading/competitor/neutral = 1
thin seed: Consulting/app-website/positive = 0
thin seed: Consulting/app-website/negative = 0
thin seed: Consulting/app-website/neutral = 0
thin seed: Consulting/competitor/positive = 0
thin seed: Consulting/competitor/negative = 0
thin seed: Consulting/competitor/neutral = 0


In [25]:
# plant sentiment splits per group  (group key = industry x org x aspect; each org its own split)
# MAX_REVIEWS = 12000
# MIN_PER_GROUP = 60
# small sample test:
MAX_REVIEWS = 120
MIN_PER_GROUP = 9

GROUPS, gid = [], 1
raw_sizes = []
for ind in INDUSTRIES:
    for org in ORGS[ind]:
        for asp in ASPECTS:
            pos = round(random.uniform(0.20, 0.85), 2)
            neu = round(random.uniform(0.02, 0.06), 2)
            neg = round(1 - pos - neu, 4)
            raw_sizes.append((ind, org, asp, {"positive": pos, "negative": neg, "neutral": neu},
                              random.randint(45, 65)))

# guarantee minimum, distribute remaining budget proportionally
floor_total = MIN_PER_GROUP * len(raw_sizes)
remaining = MAX_REVIEWS - floor_total
raw_total = sum(r[4] for r in raw_sizes)
for ind, org, asp, shares, raw_n in raw_sizes:
    bonus = int(round(raw_n / raw_total * remaining)) if remaining > 0 else 0
    GROUPS.append((f"G{gid:04d}", ind, org, asp, shares, MIN_PER_GROUP + bonus))
    gid += 1
GRP = {g[0]: g for g in GROUPS}

print(f"groups: {len(GROUPS)} | budget: {MAX_REVIEWS} | actual: {sum(g[5] for g in GROUPS)} | min group: {min(g[5] for g in GROUPS)}")
for g in GROUPS[:3]:
    print(f"{g[0]}  {g[1]:22s} {g[2]:9s} {g[3]:18s}  pos={g[4]['positive']:.0%}  neg={g[4]['negative']:.0%}  neu={g[4]['neutral']:.0%}  n={g[5]}")

groups: 8 | budget: 120 | actual: 120 | min group: 14
G0001  Trading                TradeA    app-website         pos=46%  neg=52%  neu=2%  n=14
G0002  Trading                TradeA    competitor          pos=27%  neg=69%  neu=4%  n=15
G0003  Trading                TradeB    app-website         pos=72%  neg=26%  neu=2%  n=16


In [26]:
# expand groups into cells (exact counts)
def expand(g):
    _, ind, org, asp, sh, n = g
    c = {s: int(round(sh.get(s, 0) * n)) for s in SENTS}
    r = n - sum(c.values())
    if r: c[max(c, key=c.get)] += r      # push rounding residue onto largest bucket
    return [(ind, org, asp, s, c[s], g[0]) for s in SENTS if c[s] > 0]

CELLS = [c for g in GROUPS for c in expand(g)]
print(len(GROUPS), "groups ->", len(CELLS), "cells ->", sum(c[4] for c in CELLS), "reviews to generate")

8 groups -> 21 cells -> 120 reviews to generate


In [27]:
# save planted insights: 1 row per org group + 1 row per industry group (deduped).
#   org group      = (industry, org, aspect) -> that org's planted split
#   industry group = (industry, aspect)      -> volume-weighted aggregate of its orgs
ind_totals = defaultdict(lambda: {"positive": 0.0, "negative": 0.0, "neutral": 0.0, "n": 0})
for _gid, ind, org, asp, sh, n in GROUPS:
    agg = ind_totals[(ind, asp)]
    for s in SENTS:
        agg[s] += sh[s] * n
    agg["n"] += n

org_rows = [{"group_id": _gid, "level": "org", "industry": ind, "org": org, "child_aspect": asp,
             "pos_share": round(sh["positive"], 3), "neg_share": round(sh["negative"], 3),
             "neu_share": round(sh["neutral"], 3), "n": n}
            for _gid, ind, org, asp, sh, n in GROUPS]

industry_rows = [{"group_id": "", "level": "industry", "industry": ind, "org": "ALL", "child_aspect": asp,
                  "pos_share": round(a["positive"] / a["n"], 3), "neg_share": round(a["negative"] / a["n"], 3),
                  "neu_share": round(a["neutral"] / a["n"], 3), "n": a["n"]}
                 for (ind, asp), a in ind_totals.items()]

INSIGHTS = pd.DataFrame(org_rows + industry_rows)
insights_path = OUT_DIR / f"{RUN_TS}_insights.csv"
INSIGHTS.to_csv(insights_path, index=False)
print(f"wrote {insights_path.name}: {len(org_rows)} org rows + {len(industry_rows)} industry rows = {len(INSIGHTS)}")
INSIGHTS.head()

wrote 2026-07-22_184209_insights.csv: 8 org rows + 4 industry rows = 12


,group_id,level,industry,org,child_aspect,pos_share,neg_share,neu_share,n
0,G0001,org,Trading,TradeA,app-website,0.46,0.52,0.02,14
1,G0002,org,Trading,TradeA,competitor,0.27,0.69,0.04,15
2,G0003,org,Trading,TradeB,app-website,0.72,0.26,0.02,16
3,G0004,org,Trading,TradeB,competitor,0.41,0.55,0.04,15
4,G0005,org,Consulting,ConsultA,app-website,0.74,0.23,0.03,14


# Step 2: generate reviews according to the pre-defined patterns

In [28]:
def fabsa_seeds(ind, asp, sent, k=3):
    m = FABSA[(FABSA.industry==ind) & (FABSA.child_aspect==asp) & (FABSA.sentiment==sent)]
    return m.text.dropna().drop_duplicates().head(k).tolist()

# generate reviews for one (industry, org, aspect, sentiment) cell.
# returns (named, plain): `named` mention the org by name, `plain` mention no company at all.
def gen_reviews(ind, org, asp, sent, n_named, n_plain, seeds):
    ex = "\n".join(f"- {s}" for s in seeds) or "- (none)"
    out = gemini_json(
      f"Write short customer reviews for '{org}', a {ind} company, each expressing "
      f"{sent.upper()} sentiment about '{asp}'. Match the style of these real examples:\n{ex}\n"
      f"Rules: each review MUST be 5-30 words and vary the wording. Return ONLY a JSON object with:\n"
      f"  'named': a list of {n_named} reviews that naturally mention the company by name '{org}'.\n"
      f"  'plain': a list of {n_plain} reviews that do NOT mention any company or brand name.",
      block="generate")
    if not isinstance(out, dict):
        out = {"named": [], "plain": out if isinstance(out, list) else []}
    named = [r for r in out.get("named", []) if 5 <= len(r.split()) <= 30 and org in r]
    plain = [r for r in out.get("plain", []) if 5 <= len(r.split()) <= 30 and org not in r]
    return named, plain

In [29]:
# rewrite ~a slice of reviews to add ~10% contextual noise: a brief off-topic aside,
# casual filler, or passing mention of something unrelated — WITHOUT changing the
# labelled aspect/sentiment or adding/removing any company name.
def add_noise(reviews, asp, sent):
    if not reviews:
        return []
    out = gemini_json(
      f"Rewrite each review so that roughly 10% of the text is incidental context — a brief "
      f"off-topic aside, casual filler, or a passing mention of something unrelated. The main "
      f"point must remain a clear {sent.upper()} opinion about '{asp}'. Do NOT add or remove any "
      f"company or brand names, and do NOT change the opinion. Keep each rewrite 8-45 words. "
      f"Return ONLY a JSON list of strings, in the same order.\n{json.dumps(reviews)}",
      block="noise")
    return out if isinstance(out, list) else reviews

# independent Gemini call to verify aspect and sentiment labels
def verify(reviews, asp, sent):
    if not reviews:
        return []
    return gemini_json(
      f"For each review answer true only if it expresses {sent.upper()} sentiment about "
      f"'{asp}', else false. Return ONLY a JSON list of booleans, same order.\n"
      f"{json.dumps(reviews)}", block="verify")

In [30]:
# Build the corpus (slow). Per cell, three phases:
#   1. generate_cell  -> named + plain reviews, verified, topped up once
#   2. inject_noise   -> rewrite ~10% to add contextual noise, re-verified
#   3. assemble       -> flatten into rows
start = time.time()
NAME_FRAC  = 0.20   # ~20% of each cell's reviews mention the org by name
NOISE_FRAC = 0.10   # ~10% of each cell's reviews get contextual noise


def generate_cell(ind, org, asp, sent, n):
    """Return up to n verified reviews for one cell: ~NAME_FRAC mention the org, rest don't."""
    n_named = round(NAME_FRAC * n)
    n_plain = n - n_named
    seeds = fabsa_seeds(ind, asp, sent)
    named, plain = [], []

    for _ in range(2):  # generate, then one top-up attempt for any shortfall
        need_named = n_named - len(named)
        need_plain = n_plain - len(plain)
        if need_named <= 0 and need_plain <= 0:
            break
        gn, gp = gen_reviews(ind, org, asp, sent, need_named + 1, need_plain + 1, seeds)
        candidates = [(r, True) for r in gn] + [(r, False) for r in gp]
        ok = verify([r for r, _ in candidates], asp, sent)
        for (r, _), good in zip(candidates, ok):
            if not good:
                continue
            if org in r and len(named) < n_named:
                named.append(r)
            elif org not in r and len(plain) < n_plain:
                plain.append(r)

    return named[:n_named] + plain[:n_plain]


def inject_noise(reviews, org, asp, sent):
    """Rewrite ~NOISE_FRAC of the reviews with contextual noise; keep a rewrite only if it
    still verifies and preserves the org-mention status and 5-45 word length."""
    n_noisy = round(NOISE_FRAC * len(reviews))
    if n_noisy == 0:
        return reviews
    # spread the chosen indices evenly across the cell
    step = len(reviews) / n_noisy
    idx = sorted({int(j * step) for j in range(n_noisy)})

    rewritten = add_noise([reviews[j] for j in idx], asp, sent)
    ok = verify([str(r) for r in rewritten], asp, sent)
    for j, new, good in zip(idx, rewritten, ok):
        new = str(new)
        keeps_name = (org in reviews[j]) == (org in new)
        if good and keeps_name and 5 <= len(new.split()) <= 45:
            reviews[j] = new
    return reviews


In [ ]:
# ---- corpus generation COMMENTED OUT for pipeline testing ----
# Skip the slow LLM corpus build; load the most recent saved corpus instead, so the rest of the
# pipeline (Step 3 tasks -> gold answers -> verify -> questions -> answers -> save) can run.
# To regenerate the corpus, comment out the load block at the bottom and uncomment this:
#
# rows, rid = [], 0
# for i, (ind, org, asp, sent, n, gid) in enumerate(CELLS, 1):
#     kept = generate_cell(ind, org, asp, sent, n)
#     kept = inject_noise(kept, org, asp, sent)
#     for text in kept:
#         rows.append((rid, ind, org, asp, sent, text, gid)); rid += 1
#     print(f"[{i}/{len(CELLS)}] {gid} {org} {sent}: {len(kept)}/{n}")
#
# CORPUS = pd.DataFrame(rows, columns=["review_id","industry","org","child_aspect","sentiment","text","group_id"])
# corpus_path = OUT_DIR / f"{RUN_TS}_corpus.csv"
# CORPUS.to_csv(corpus_path, index=False)
#
# elapsed = time.time() - start
# mention = CORPUS.apply(lambda x: x["org"] in x["text"], axis=1).mean() if len(CORPUS) else 0
# print(f"\ncorpus: {len(CORPUS)} reviews ({mention:.0%} name their org) in {elapsed/60:.1f} min -> {corpus_path.name}")
# cost_report()

# --- load the most recent saved corpus (test the rest of the pipeline) ---
_corpus_path = max(OUT_DIR.glob("*corpus.csv"), key=lambda p: p.stat().st_mtime)
CORPUS = pd.read_csv(_corpus_path)
print(f"loaded existing corpus: {_corpus_path.name}  ({len(CORPUS)} reviews)")
print("industries:", sorted(CORPUS.industry.unique()), "| orgs:", sorted(CORPUS.org.unique()))

[1/21] G0001 TradeA positive: 6/6
[2/21] G0001 TradeA negative: 8/8
[3/21] G0002 TradeA positive: 1/4
[4/21] G0002 TradeA negative: 3/10
[5/21] G0002 TradeA neutral: 1/1
[6/21] G0003 TradeB positive: 12/12
[7/21] G0003 TradeB negative: 4/4
[8/21] G0004 TradeB positive: 0/6
[9/21] G0004 TradeB negative: 8/8
[10/21] G0004 TradeB neutral: 1/1
[11/21] G0005 ConsultA positive: 11/11
[12/21] G0005 ConsultA negative: 3/3
[13/21] G0006 ConsultA positive: 5/5
[14/21] G0006 ConsultA negative: 10/10
[15/21] G0006 ConsultA neutral: 1/1
[16/21] G0007 ConsultB positive: 8/8
[17/21] G0007 ConsultB negative: 6/6
[18/21] G0007 ConsultB neutral: 1/1
[19/21] G0008 ConsultB positive: 6/6
[20/21] G0008 ConsultB negative: 8/8
[21/21] G0008 ConsultB neutral: 1/1

corpus: 104 reviews (22% name their org) in 7.9 min -> 2026-07-22_184209_corpus.csv
generate     calls= 27  in=   4311  out=  30970  $0.0787
verify       calls= 37  in=   4725  out=  15749  $0.0408
noise        calls= 10  in=   1113  out=   8305  $0

In [32]:
CORPUS.head()

,review_id,industry,org,child_aspect,sentiment,text,group_id
0,0,Trading,TradeA,app-website,positive,"TradeA's app is simply fantastic! Honestly, it's incredibly intuitive and reliable, a real game-changer.",G0001
1,1,Trading,TradeA,app-website,positive,"Fantastic app, very smooth and reliable.",G0001
2,2,Trading,TradeA,app-website,positive,"User-friendly interface, makes trading so much easier.",G0001
3,3,Trading,TradeA,app-website,positive,Excellent website! Navigating is a breeze.,G0001
4,4,Trading,TradeA,app-website,positive,This platform is incredibly intuitive and performs perfectly.,G0001


# Step 3: define tasks

## Build solver functions

In [33]:
# build solver functions for each task.
# each solver runs the tools on the corpus to produce the gold answer + records the tool calls.
# `where` selects a segment: {"industry": name} or {"org": name}. filter is implicit (not a step).

# ---- 1-step solvers (descriptive) ----

def solve_count(where, asp, sent):
    # share_of returns {n, count, share}; read the count for a "how many" answer
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["count"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

def solve_share(where, asp, sent):
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["share"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

# ---- 2-step solvers ----

def solve_top_k(where, k, by):
    filt = {**where, **({"sentiment":"negative"} if by=="negative" else {})}
    top = rank_top(count_by(filter_data(CORPUS, **filt), "child_aspect"), by="count", top_n=k)
    return top.child_aspect.tolist(), [
        {"tool":"count_by","args":{**filt,"group_by":"child_aspect"}},
        {"tool":"rank_top","args":{"by":"count","top_n":k}}]

def solve_split(where, group_by):
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), group_by))
    v = {k: float(sh[k].iloc[0]) for k in ["pos_share","neg_share","neu_share"]}
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":group_by}},
        {"tool":"add_shares","args":{}}]

def solve_compare(where_a, where_b, asp, sent):
    ra = share_of(filter_data(CORPUS, **where_a, child_aspect=asp), sent)
    rb = share_of(filter_data(CORPUS, **where_b, child_aspect=asp), sent)
    return {"a":ra["share"],"b":rb["share"],"higher":"a" if (ra["share"] or 0)>(rb["share"] or 0) else "b"}, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}}]

# ---- 3-step solver ----

def solve_two_prop(where_a, where_b, asp, sent):
    da = filter_data(CORPUS, **where_a, child_aspect=asp)
    db = filter_data(CORPUS, **where_b, child_aspect=asp)
    r = two_prop_test(da, db, sent)
    return r, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}},
        {"tool":"two_prop_test","args":{"sentiment":sent}}]

# ---- 4-step solvers (diagnostic) ----

def solve_driver(where, k):
    # L6: rank child aspects by negative share (severity), with a volume guard
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect"))
    vol = apply_min_volume(sh, min_volume=5)
    top = rank_top(vol, by="neg_share", top_n=k)
    v = list(zip(top.child_aspect, top.neg_share.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"apply_min_volume","args":{"min_volume":5}},
        {"tool":"rank_top","args":{"by":"neg_share","top_n":k}}]

def solve_prioritise(where, k):
    # L7: rank child aspects by priority = complaint volume x severity (prioritisation quadrant)
    sh = add_priority(add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect")))
    top = rank_top(sh, by="priority", top_n=k)
    v = list(zip(top.child_aspect, top.priority.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"add_priority","args":{}},
        {"tool":"rank_top","args":{"by":"priority","top_n":k}}]

SOLVERS = {
    "solve_count": solve_count, "solve_share": solve_share,
    "solve_top_k": solve_top_k, "solve_split": solve_split,
    "solve_compare": solve_compare, "solve_two_prop": solve_two_prop,
    "solve_driver": solve_driver, "solve_prioritise": solve_prioritise,
} 

Task structure:
- tid: task ID, numbered by increasing difficulty
- type: question intent (descriptive, diagnostic or prescriptive)
- inds: industries
- solver: function that computes the gold answer 
- args: arguments to pass to that solver
- spec: structured description of the task, input to Gemini to write the question
- steps: number of tool calls in the gold path

## Create task specs

In [34]:
# generate task specs — 3 tasks per level, 10 levels => 30 tasks total.
#   Segment = `where` dict {"industry": name} or {"org": name}; filter is implicit.
#   industry-level: L1-L5 (per_level each).   org-level: L6-L10 (one per focus org = ORGS[ind][0]).
#   Per the taxonomy, L6-L10 are organisation-specific (one org); the industry is used only as an
#   internal "industry average" reference inside L8/L10. Prescriptive tasks (L8/L9/L10) compose
#   `components` = a list of {key, solver, args, spec, steps}; the builder runs each to get its
#   finding + tool path.
TASKS_PER_LEVEL = 3

def auto_tasks(industries, aspects, seed=42, per_level=TASKS_PER_LEVEL):
    rng = random.Random(seed)
    T = []

    def _add(tid, typ, inds, solver, args, spec, steps):
        T.append((tid, typ, set(inds), solver, args, spec, steps))

    def _comp(key, solver, args, op, steps):
        return {"key": key, "solver": solver, "args": args, "spec": {"op": op}, "steps": steps}

    def _compose(tid, ind, fo, op, components, spec_extra=None):
        steps = sum(c["steps"] for c in components) + 1   # + sample_reviews evidence
        spec = {"op": op, "scope": "org", "subject": fo, "where": {"org": fo}, **(spec_extra or {})}
        _add(tid, "prescriptive", [ind], "solve_compose", (components,), spec, steps)

    pairs = list(itertools.combinations(industries, 2))
    sents = ["positive", "negative"]

    # ============ INDUSTRY-LEVEL: L1-L5 (3 each) ============
    di = gi = 0

    # L1 count / share
    pool = [(ind, asp) for ind in industries for asp in aspects]
    rng.shuffle(pool)
    l1_ops = [("count", "negative"), ("share", "positive"), ("share", "negative")]
    for j in range(per_level):
        ind, asp = pool[j % len(pool)]
        op, sent = l1_ops[j % len(l1_ops)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], f"solve_{op}", ({"industry": ind}, asp, sent),
             {"op": op, "scope": "industry", "subject": ind, "aspect": asp, "sent": sent}, 1)

    # L2 split
    for j in range(per_level):
        ind = industries[j % len(industries)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_split", ({"industry": ind}, "industry"),
             {"op": "split", "scope": "industry", "subject": ind}, 2)

    # L3 top_k
    l3 = [("volume", 1), ("negative", 3), ("negative", 2)]
    for j in range(per_level):
        ind = industries[j % len(industries)]
        by, k = l3[j % len(l3)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_top_k", ({"industry": ind}, k, by),
             {"op": f"top_{by}", "scope": "industry", "subject": ind, **({"k": k} if k > 1 else {})}, 2)

    # L4 compare
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = sents[j % len(sents)]
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_compare",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "compare", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 2)

    # L5 two_prop
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = rng.choice(sents)
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_two_prop",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "test", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 3)

    # ============ ORG-LEVEL: L6-L10 (organisation-specific, one focus org per industry) ============
    opi = 0   # DIAG counter `gi` continues from L4/L5 so org-level DIAG ids stay unique
    for ind in industries:
        fo = ORGS[ind][0]                                  # focus org (organisation-specific)
        wfo, wind = {"org": fo}, {"industry": ind}

        # L6 driver identification
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [ind], "solve_driver", (wfo, 2),
             {"op": "driver", "scope": "org", "subject": fo, "k": 2}, 4)
        # L7 prioritisation (volume x severity)
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [ind], "solve_prioritise", (wfo, 3),
             {"op": "prioritise", "scope": "org", "subject": fo, "k": 3}, 4)

        # L8 comparative diagnosis: org drivers vs the industry-average drivers
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "compare_diag", [
            _comp("org_drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("industry_avg_drivers", "solve_driver", (wind, 3), "driver", 4),
        ], {"industry": ind})
        # L9 targeted recommendation: overview + gap vs the industry average
        asp9, sent9 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "recommend", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp9, sent9), "test", 3),
        ])
        # L10 full CX report: multiple descriptive + multiple diagnostic
        asp10, sent10 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "report", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("top_complaints", "solve_top_k", (wfo, 3, "negative"), "top_negative", 2),
            _comp("drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("priorities", "solve_prioritise", (wfo, 3), "prioritise", 4),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp10, sent10), "test", 3),
        ])

    return T

T = auto_tasks(INDUSTRIES, ASPECTS, seed=42)
n_ind = sum(t[5].get("scope") == "industry" for t in T)
n_org = sum(t[5].get("scope") == "org" for t in T)
print(f"{len(T)} tasks ({TASKS_PER_LEVEL}/level x 10 levels): {n_ind} industry (L1-L5) + {n_org} org (L6-L10)\n")
for tid, typ, inds, solver, args, spec, steps in T:
    print(f"{tid:10s} {steps:2d}-step  {spec['scope']:8s} {typ:12s}  {spec['op']:12s} subj={spec.get('subject') or spec.get('a')}")

25 tasks (3/level x 10 levels): 15 industry (L1-L5) + 10 org (L6-L10)

DESC-01     1-step  industry descriptive   count        subj=Consulting
DESC-02     1-step  industry descriptive   share        subj=Trading
DESC-03     1-step  industry descriptive   share        subj=Consulting
DESC-04     2-step  industry descriptive   split        subj=Trading
DESC-05     2-step  industry descriptive   split        subj=Consulting
DESC-06     2-step  industry descriptive   split        subj=Trading
DESC-07     2-step  industry descriptive   top_volume   subj=Trading
DESC-08     2-step  industry descriptive   top_negative subj=Consulting
DESC-09     2-step  industry descriptive   top_negative subj=Trading
DIAG-01     2-step  industry diagnostic    compare      subj=Trading
DIAG-02     2-step  industry diagnostic    compare      subj=Trading
DIAG-03     2-step  industry diagnostic    compare      subj=Trading
DIAG-04     3-step  industry diagnostic    test         subj=Trading
DIAG-05     3-step  

# Step 4: derive gold answers (JSON) and gold tool paths

In [35]:
# run solvers to get gold answers and paths (descriptive and diagnostic)
records = {}
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name == "solve_compose":
        continue
    fn = SOLVERS[solver_name]
    ans, path = fn(*args)
    assert len(path) == steps, f"{tid}: expected {steps} steps, got {len(path)}"
    records[tid] = dict(task_id=tid, type=type, steps=steps,
                        industries=sorted(inds), spec=spec,
                        gold_answer=ans, gold_tool_path=path)

for tid in sorted(records)[:5]:
    print(f"{tid} ({records[tid]['steps']}-step): {records[tid]['gold_answer']}")

DESC-01 (1-step): 9
DESC-02 (1-step): 0.071
DESC-03 (1-step): 0.581
DESC-04 (2-step): {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
DESC-05 (2-step): {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}


In [36]:
# build prescriptive tasks (L8 comparative diagnosis, L9 recommendation, L10 full report).
# run each component analysis, concatenate their tool paths, then sample_reviews for evidence.
# gold_answer = {"findings": {component_key: answer}}; sampled reviews are evidence, not graded.
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name != "solve_compose":
        continue
    components = args[0]
    path, findings, meta = [], {}, []
    for c in components:
        ans, cpath = SOLVERS[c["solver"]](*c["args"])
        assert len(cpath) == c["steps"], f"{tid}/{c['key']}: {len(cpath)} != {c['steps']}"
        findings[c["key"]] = ans
        path += cpath
        meta.append({"key": c["key"], "spec": c["spec"], "steps": c["steps"]})
    path += [{"tool": "sample_reviews", "args": {**spec["where"], "sentiment": "negative", "n": 3}}]
    assert len(path) == steps, f"{tid}: {len(path)} != {steps}"
    records[tid] = dict(task_id=tid, type=type, steps=steps, industries=sorted(inds),
                        spec={**spec, "components": meta},
                        gold_answer={"findings": findings}, gold_tool_path=path)
    print(f"{tid} ({steps}-step, {spec['op']}): {[c['key'] for c in components]} + sample_reviews")

PRES-01 (9-step, compare_diag): ['org_drivers', 'industry_avg_drivers'] + sample_reviews
PRES-02 (6-step, recommend): ['overview', 'vs_industry_avg'] + sample_reviews
PRES-03 (16-step, report): ['overview', 'top_complaints', 'drivers', 'priorities', 'vs_industry_avg'] + sample_reviews
PRES-04 (9-step, compare_diag): ['org_drivers', 'industry_avg_drivers'] + sample_reviews
PRES-05 (6-step, recommend): ['overview', 'vs_industry_avg'] + sample_reviews
PRES-06 (16-step, report): ['overview', 'top_complaints', 'drivers', 'priorities', 'vs_industry_avg'] + sample_reviews


## Verify gold tool paths and answers

Sanity check: independently **re-run each recorded `gold_tool_path`** against `CORPUS`
(calling the real tool functions directly, *not* the solver functions that produced them) and
confirm it reproduces `gold_answer`. A wrong tool name or argument in a path would make the two
diverge. The final answer is read off the last tool's output, shaped per the task's `spec["op"]`.

In [37]:
# Re-run every recorded gold_tool_path against the in-memory CORPUS with the REAL tool functions
# (not the solvers that produced them) and check it reproduces gold_answer. A wrong tool name or
# argument would make the two diverge.
# We verify the `records` generated just above (Steps 3-4), NOT a reloaded tasks.csv: this cell
# runs before Step 6 writes the CSV, so glob-loading would pick up a PRIOR run's file (possibly
# with retired tool names). `question` is added later in Step 5, so it's optional here.
#   count -> share_of[...]["count"]   share -> share_of[...]["share"]
#   prescriptive (recommend/compare_diag/report): findings read by slicing the path per component.
import inspect
import tools as _tools_mod

# guard: every tool named in a gold_tool_path must be a real tool defined in the tools module.
REAL_TOOLS = {n for n, f in inspect.getmembers(_tools_mod, inspect.isfunction)
              if f.__module__ == _tools_mod.__name__ and not n.startswith("_")}
for _tid, _rec in records.items():
    for _step in _rec["gold_tool_path"]:
        assert _step["tool"] in REAL_TOOLS, (
            f"{_tid}: gold_tool_path references {_step['tool']!r}, which is not a real tool in the "
            f"tools module. Real tools: {sorted(REAL_TOOLS)}")
print(f"tool-name check passed: every path uses only real tools ({len(REAL_TOOLS)} available)")

def _slice(args, keys):
    return filter_data(CORPUS, **{k: args[k] for k in keys if k in args})

# key sets exclude 'sentiment' where a tool must NOT pre-filter by sentiment (share_of, breakdown)
_SEG = ("industry", "org", "child_aspect")               # segment slice (no sentiment)
_SEG_S = ("industry", "org", "child_aspect", "sentiment")  # segment slice incl. sentiment
_COMPOSE_OPS = ("recommend", "compare_diag", "report")

def run_gold_path(path, spec, records):
    op = spec["op"]

    # prescriptive: findings = each component's answer, read by slicing the path per component
    if op in _COMPOSE_OPS:
        findings, idx = {}, 0
        for comp in spec["components"]:
            n = comp["steps"]
            findings[comp["key"]] = run_gold_path(path[idx:idx + n], comp["spec"], records)
            idx += n
        return {"findings": findings}

    cur = None                            # current DataFrame for chained ops
    share_slices, share_results = [], []  # from share_of steps -> feed count/share/compare/two_prop
    tp = None
    for step in path:
        t, a = step["tool"], step["args"]
        if t == "share_of":
            df = _slice(a, _SEG)
            share_slices.append(df)
            share_results.append(share_of(df, a["sentiment"]))
        elif t == "count_by":
            cur = count_by(_slice(a, _SEG_S + ("parent_aspect",)), a["group_by"])
        elif t == "sentiment_breakdown":
            cur = sentiment_breakdown(_slice(a, _SEG), a["group_by"])
        elif t == "add_shares":
            cur = add_shares(cur)
        elif t == "add_priority":
            cur = add_priority(cur)
        elif t == "apply_min_volume":
            cur = apply_min_volume(cur, min_volume=a.get("min_volume", 30))
        elif t == "rank_top":
            cur = rank_top(cur, by=a["by"], top_n=a["top_n"])
        elif t == "two_prop_test":
            tp = two_prop_test(share_slices[-2], share_slices[-1], a["sentiment"])
        elif t == "sample_reviews":
            sample_reviews(_slice(a, _SEG_S), n=a.get("n", 3))  # evidence only
        else:
            raise ValueError(f"unknown tool in path: {t}")

    # read the final answer off the last tool's output, shaped per task op
    if op == "count":
        return share_results[-1]["count"]
    if op == "share":
        return share_results[-1]["share"]
    if op in ("top_volume", "top_negative"):
        return cur["child_aspect"].tolist()
    if op == "split":
        return {k: float(cur[k].iloc[0]) for k in ("pos_share", "neg_share", "neu_share")}
    if op == "compare":
        a_, b_ = share_results[0]["share"], share_results[1]["share"]
        return {"a": a_, "b": b_, "higher": "a" if (a_ or 0) > (b_ or 0) else "b"}
    if op == "test":
        return tp
    if op == "driver":
        return [[asp, float(round(ns, 4))] for asp, ns in zip(cur["child_aspect"], cur["neg_share"])]
    if op == "prioritise":
        return [[asp, float(round(p, 4))] for asp, p in zip(cur["child_aspect"], cur["priority"])]
    raise ValueError(f"unknown op: {op}")


def _match(a, b, tol=1e-4):
    """Tolerant structural comparison: list==tuple, floats within tol, exact otherwise."""
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(float(a), float(b), abs_tol=tol)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_match(a[k], b[k], tol) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_match(x, y, tol) for x, y in zip(a, b))
    return a == b


print(f"Verifying {len(records)} gold tool paths against CORPUS ({len(CORPUS)} reviews)")
print("=" * 78)
n_pass = 0
for tid, rec in records.items():
    got = run_gold_path(rec["gold_tool_path"], rec["spec"], records)
    ok = _match(got, rec["gold_answer"])
    n_pass += ok
    print(f"\n[{tid}] {rec['steps']}-step  {rec.get('question', '(question written in Step 5)')}")
    print(f"  gold answer : {rec['gold_answer']}")
    print(f"  path re-ran : {got}")
    print(f"  {'PASS' if ok else 'FAIL <<<<<<'}")

print("\n" + "=" * 78)
print(f"{n_pass}/{len(records)} tasks: gold_tool_path reproduces gold_answer")

tool-name check passed: every path uses only real tools (12 available)
Verifying 25 gold tool paths against CORPUS (104 reviews)

[DESC-01] 1-step  (question written in Step 5)
  gold answer : 9
  path re-ran : 9
  PASS

[DESC-02] 1-step  (question written in Step 5)
  gold answer : 0.071
  path re-ran : 0.071
  PASS

[DESC-03] 1-step  (question written in Step 5)
  gold answer : 0.581
  path re-ran : 0.581
  PASS

[DESC-04] 2-step  (question written in Step 5)
  gold answer : {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
  path re-ran : {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
  PASS

[DESC-05] 2-step  (question written in Step 5)
  gold answer : {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}
  path re-ran : {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}
  PASS

[DESC-06] 2-step  (question written in Step 5)
  gold answer : {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
  path re-ran : {'pos_share': 0.432, 'neg_share

# Step 5: generate task questions

In [38]:
# use LLM to write questions from task specs
start = time.time()
def brief(sp):
    o = sp["op"]; who = sp.get("subject")
    if o == "count":        return f"How many complaints {who} received about {sp['aspect']}."
    if o == "share":        return f"The share of {who} feedback on {sp['aspect']} that is {'praise' if sp['sent']=='positive' else 'complaints'}."
    if o == "top_volume":   return f"Which single topic {who} customers mention most."
    if o == "top_negative": return f"The {sp['k']} topics with the most complaints for {who}."
    if o == "split":        return f"The overall breakdown of happy vs unhappy {who} customers."
    if o == "compare":      return f"Whether {sp['a']} or {sp['b']} customers have more {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']}."
    if o == "test":         return f"Whether the difference in {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']} between {sp['a']} and {sp['b']} is statistically significant."
    if o == "driver":       return (f"The single biggest driver of dissatisfaction for {who}." if sp['k']==1
                                    else f"The top {sp['k']} drivers of dissatisfaction for {who}.")
    if o == "prioritise":   return f"The top {sp['k']} issues {who} should fix first, weighing both how many customers are affected and how unhappy they are."
    if o == "compare_diag": return f"How {who}'s biggest complaint drivers compare with the {sp['industry']} industry average, and where {who} does worse."
    if o == "recommend":    return f"Diagnose the biggest customer problems for {who} and recommend the top fixes."
    if o == "report":       return f"A full customer-experience report for {who}: overall happiness, top complaints, biggest drivers, what to prioritise, and how it compares to the industry average."

briefs = {tid: brief(r["spec"]) for tid, r in records.items()}

qs = gemini_json(
  "You are a CX analytics lead writing questions for a business intelligence tool.\n\n"
  "For each item below, write ONE clear, specific, realistic question that a business "
  "stakeholder would  ask. The question must be answerable using exactly the "
  "analysis described in the item.\n\n"

  "LANGUAGE:\n"
  "- Use plain, professional business English.\n"
  "- Talk about 'complaints', 'issues', 'what customers like', 'praise', 'frustrations' etc.\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'          -> 'the app or website'\n"
  "- 'ease-of-use'          -> 'how easy the service is to use'\n"
  "- 'account-access'       -> 'accessing their account'\n"
  "- 'price-value-for-money'-> 'pricing and value for money'\n"
  "(Apply the same natural-language treatment to any other topic.)\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Some items refer to a whole industry (e.g. 'Banking'); others to a specific company "
  "within an industry (e.g. 'BankA', 'ITA'). Always keep the exact name given.\n"
  "- When comparing one company against the rest of its industry, use the phrase "
  "'the industry average'.\n"
  "- Preserve any specific counts, such as 'top 3'.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its question string. Use the ids EXACTLY "
  "as given, and include every id.\n\n"
  + json.dumps(briefs), block="questions", max_tokens=6000)  # one question per task; give the
  # same headroom as the answers call so a large batch's JSON isn't truncated (-> parse failure)

# map each generated question back onto its task by id. Iterate over `records` (not qs) so a
# stray/renamed id from the model can't KeyError here, and surface any task it skipped -- an
# unfilled `question` would otherwise blow up downstream in Step 6.
missing = [tid for tid in records if tid not in qs]
if missing:
    raise ValueError(f"question generation returned no question for {len(missing)} task(s): {missing}")
for tid in records:
    records[tid]["question"] = qs[tid]
    print(f"{tid}: {qs[tid]}")

elapsed = time.time() - start
print(f"\nquestions generated in {elapsed:.1f}s")
cost_report()

DESC-01: How many complaints did Consulting receive regarding the app or website?
DESC-02: What share of Trading feedback expresses praise for a competitor?
DESC-03: What share of Consulting feedback expresses complaints about a competitor?
DESC-04: What is the overall breakdown of happy versus unhappy Trading customers?
DESC-05: What is the overall breakdown of happy versus unhappy Consulting customers?
DESC-06: What is the overall breakdown of happy versus unhappy Trading customers?
DESC-07: Which single topic do Trading customers mention most frequently?
DESC-08: What are the top 3 topics generating the most complaints for Consulting?
DESC-09: What are the top 2 topics generating the most complaints for Trading?
DIAG-01: Do Trading or Consulting customers express more praise for the app or website?
DIAG-02: Do Trading or Consulting customers express more complaints about a competitor?
DIAG-03: Do Trading or Consulting customers express more praise for the app or website?
DIAG-04: Is

# Step 6: generate gold answers (text)

In [39]:
# # only needed after restarting kernel
# # Load the saved tasks export and rebuild the minimal structures expected by Step 6.
# # This lets the question-generation and answer-generation cells run without executing the earlier cells.

# tasks = pd.read_csv("../benchmark_outputs/tasks.csv")

# # Rebuild the lightweight records structure expected by the notebook.
# records = {}
# for _, row in tasks.iterrows():
#     tid = row["task_id"]
#     records[tid] = {
#         "task_id": tid,
#         "question": row.get("question", ""),
#         "gold_answer": row.get("gold_answer"),
#         "gold_tool_path": row.get("gold_tool_path"),
#     }

# # Rebuild a minimal T structure so the final save cell can still write tasks.csv in the same format.
# # The notebook later uses T to preserve task order and the task_id list.
# T = [(row["task_id"], None, None, None, None, None, None) for _, row in tasks.iterrows()]


In [32]:
# business-facing natural-language gold answer (generated AFTER the JSON answer + question).
# Each NL answer must use ONLY the figures/insights in the JSON gold answer and answer that task's
# question directly. Stored as records[tid]["gold_answer_nl"] -> a second gold-answer column.
start = time.time()
def _sides(rec):
    """Resolve the 'a'/'b' placeholders in a comparison gold answer to real entity names, so the
    model can't invert 'which side is higher'. Only compare/test (flat) and the org-vs-industry
    'vs_industry_avg' finding in recommend/report use a/b; each task has at most one such compare."""
    sp = rec["spec"]; op = sp["op"]
    if op in ("compare", "test"):                       # a and b are the two industries compared
        return {"a": sp["a"], "b": sp["b"]}
    if op in ("recommend", "report"):                   # a = the focus org, b = its industry average
        ind = rec["industries"][0] if rec.get("industries") else "the"
        return {"a": sp["subject"], "b": f"the {ind} industry average"}
    return {}

nl_inputs = {}
for tid, r in records.items():
    item = {"question": r["question"], "gold_answer": r["gold_answer"]}
    sides = _sides(r)                                   # only present for tasks with an a/b comparison
    if sides:
        item["sides"] = sides
    nl_inputs[tid] = item

nl = gemini_json(
  "You are a CX analytics lead writing the answer a stakeholder receives.\n\n"
  "For each item you are given the stakeholder's QUESTION and the correct ANSWER as "
  "structured JSON (the ground truth). Write ONE clear, business-facing answer that "
  "directly answers the question.\n\n"

  "MOST IMPORTANT RULE:\n"
  "Use ONLY the figures and findings in the JSON answer. Never invent numbers, add "
  "detail, or state anything the JSON does not support. If it isn't in the JSON, it "
  "does not go in the answer.\n\n"

  "LANGUAGE:\n"
  "- Plain, professional business English.\n"

  "NUMBERS:\n"
  "- Render proportions/shares as whole-number percentages (0.725 -> '73%').\n"
  "- Keep counts as whole numbers.\n"
  "- A 'priority' value (in prioritise / 'priorities' findings) is a 0-2 ranking score, NOT a "
  "proportion: list those items in ranked order and do NOT render the score as a percentage.\n\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'           -> 'the app or website'\n"
  "- 'ease-of-use'           -> 'how easy the service is to use'\n"
  "- 'account-access'        -> 'signing in or accessing their account'\n"
  "- 'attitude-of-staff'     -> 'staff attitude'\n"
  "- 'price-value-for-money' -> 'value for money'\n"
  "- 'discounts-promotions'  -> 'discounts and promotions'\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Use the exact company and industry names from the JSON.\n"
  "- Comparison answers label the two sides 'a' and 'b' (e.g. 'higher': 'a'; in a significance "
  "test 'p1' is side a's rate and 'p2' is side b's). A 'sides' object maps 'a' and 'b' to the real "
  "entities -- state the result using those names and take the direction ONLY from 'higher'; never "
  "guess which side is higher.\n"
  "- In a 'vs_industry_avg' finding, side 'a' is the company and side 'b' is the industry average; "
  "use the phrase 'the industry average' for side b.\n\n"

  "ANSWER SHAPE BY TASK TYPE:\n"
  "- Significance test: state whether the difference is statistically significant "
  "(p < 0.05) and which side is higher.\n"
  "- Report or recommendation: synthesise the findings into a short paragraph that "
  "ends with the single most important action to take.\n"
  "- Everything else: 1-2 sentences that answer the question directly.\n\n"

  "FORMATTING (IMPORTANT):\n"
  "- Do NOT use any quotation marks inside your answer text — no double quotes and no "
  "single quotes. Refer to features in plain words without quoting them.\n"  # quotes cause errors in JSON parsing
  "- Write plain sentences only: no markdown, no line breaks inside an answer.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its answer string. Use the ids EXACTLY "
  "as given, and include every id.\n\n"
  + json.dumps(nl_inputs, default=str), block="answers", max_tokens=16000)

# map each answer back onto its task by id. Iterate over `records` (not nl) so a stray/renamed id
# from the model can't KeyError here, and surface any task it skipped (an unfilled gold_answer_nl
# would otherwise be written to tasks.csv as an empty answer column in the next cell).
missing = [tid for tid in records if tid not in nl]
if missing:
    raise ValueError(f"answer generation returned no answer for {len(missing)} task(s): {missing}")
for tid in records:
    records[tid]["gold_answer_nl"] = nl[tid]
    print(f"{tid}: {nl[tid]}")

elapsed = time.time() - start
print(f"\nnatural-language answers generated in {elapsed:.1f}s")

DESC-01: Consulting has received 9 complaints about the app or website.
DESC-02: 7% of Trading's customer feedback mentions praise for competitors.
DESC-03: 58% of Consulting's customer feedback mentions complaints about competitors.
DESC-04: Trading customers are 43% positive, 52% negative, and 5% neutral.
DESC-05: Consulting customers are 50% positive, 45% negative, and 5% neutral.
DESC-06: Trading customers are 43% positive, 52% negative, and 5% neutral.
DESC-07: Trading customers most frequently mention the app or website.
DESC-08: The top 2 topics that generate the most complaints for Consulting are competitors and the app or website.
DESC-09: The top 2 topics that generate the most complaints for Trading are the app or website and competitors.
DIAG-01: Consulting customers express more praise for the app or website.
DIAG-02: Trading customers have more complaints about competitors.
DIAG-03: Consulting customers express more praise for the app or website.
DIAG-04: The difference i

In [33]:
# save tasks.csv and print cost report.  Pipeline outputs (all timestamp-prefixed):
#   {RUN_TS}_corpus.csv (Step 2) | {RUN_TS}_insights.csv (Step 1) | {RUN_TS}_tasks.csv (here)
def _clean(r):
    """Make record JSON-serialisable (sets -> sorted lists)."""
    r = dict(r)
    if isinstance(r.get("industries"), set):
        r["industries"] = sorted(r["industries"])
    return r

order = list(records)   # task ids in generation order (records is built from T); no need to depend on T
tasks_csv_path = OUT_DIR / f"{RUN_TS}_tasks.csv"

# two gold-answer columns -> gold_answer_json (valid JSON, for deterministic checks) and
# gold_answer_text (business-facing natural language). spec / gold_tool_path stay Python-repr.
def _csv_row(tid):
    r = _clean(records[tid])
    sp = r["spec"]                                 # break spec fields out into their own columns
    r["scope"] = sp.get("scope")                   # 'industry' or 'org'
    r["operation"] = sp.get("op")                  # e.g. count, share, driver, report
    r["industry_or_org"] = sp.get("subject")       # the industry or org the task is about
    r["gold_answer_json"] = json.dumps(r.pop("gold_answer"))
    r["gold_answer_text"] = r.pop("gold_answer_nl", None)
    return r

tasks_df = pd.DataFrame([_csv_row(tid) for tid in order])
col_order = ["task_id", "type", "steps", "scope", "operation", "industry_or_org",
             "industries", "spec", "question",
             "gold_answer_json", "gold_answer_text", "gold_tool_path"]
tasks_df = tasks_df[[c for c in col_order if c in tasks_df.columns]
                    + [c for c in tasks_df.columns if c not in col_order]]
tasks_df.to_csv(tasks_csv_path, index=False)

print(f"wrote {tasks_csv_path}")
print(f"(corpus: {OUT_DIR / (RUN_TS + '_corpus.csv')} | insights: {OUT_DIR / (RUN_TS + '_insights.csv')})\n")
cost_report()

wrote ..\benchmark_outputs\2026-07-22_200615_tasks.csv
(corpus: ..\benchmark_outputs\2026-07-22_200615_corpus.csv | insights: ..\benchmark_outputs\2026-07-22_200615_insights.csv)

questions    calls=  1  in=    877  out=   2837  $0.0074
answers      calls=  2  in=   5562  out=  17104  $0.0444
TOTAL                               $0.0518


In [35]:
tasks_df.head(3)

,task_id,type,steps,scope,operation,industry_or_org,industries,spec,question,gold_answer_json,gold_answer_text,gold_tool_path
0,DESC-01,descriptive,1,industry,count,Consulting,[Consulting],"{'op': 'count', 'scope': 'industry', 'subject': 'Consulting', 'aspect': 'app-website', 'sent': 'negative'}",How many complaints has Consulting received about the app or website?,9,Consulting has received 9 complaints about the app or website.,"[{'tool': 'share_of', 'args': {'industry': 'Consulting', 'child_aspect': 'app-website', 'sentiment': 'negative'}}]"
1,DESC-02,descriptive,1,industry,share,Trading,[Trading],"{'op': 'share', 'scope': 'industry', 'subject': 'Trading', 'aspect': 'competitor', 'sent': 'positive'}",What share of Trading's customer feedback mentions praise for competitors?,0.071,7% of Trading's customer feedback mentions praise for competitors.,"[{'tool': 'share_of', 'args': {'industry': 'Trading', 'child_aspect': 'competitor', 'sentiment': 'positive'}}]"
2,DESC-03,descriptive,1,industry,share,Consulting,[Consulting],"{'op': 'share', 'scope': 'industry', 'subject': 'Consulting', 'aspect': 'competitor', 'sent': 'negative'}",What share of Consulting's customer feedback mentions complaints about competitors?,0.581,58% of Consulting's customer feedback mentions complaints about competitors.,"[{'tool': 'share_of', 'args': {'industry': 'Consulting', 'child_aspect': 'competitor', 'sentiment': 'negative'}}]"


# End of notebook